# Parte 1 - Taller Practico y Conceptual de PDI
## Modulo A: Analisis y Ecualizacion de Histogramas (7.5%)

Universidad de Antioquia - Procesamiento Digital de Imagenes - 2026-II

Imagen de trabajo: `im1.png` (incluida junto al notebook).


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def mostrar(img, titulo, cmap=None):
    plt.figure(figsize=(7, 5))
    if img.ndim == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray', vmin=0, vmax=255)
    plt.title(titulo)
    plt.axis('off')
    plt.show()

def histograma(img, canal=0, mascara=None):
    return cv2.calcHist([img], [canal], mascara, [256], [0, 256]).flatten()

img = cv2.imread('im1.png')
assert img is not None, 'No se encontro im1.png'
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
print('Imagen:', img.shape, '| OpenCV', cv2.__version__)


---
## Ejercicio A.1 - Calculo de Histogramas

El histograma $h(r_k) = n_k$ cuenta cuantos pixeles tienen intensidad $r_k$. Normalizado por el total $N$ se obtiene la PDF $p(r_k) = n_k / N$. Se calcula para escala de grises y para cada canal en RGB y HSV.


In [ ]:
mostrar(img, 'Imagen original')

plt.figure(figsize=(8, 3))
plt.plot(histograma(gray), color='k')
plt.title('Histograma en escala de grises')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles'); plt.xlim(0, 255)
plt.grid(alpha=0.3); plt.show()

print(f'Media: {gray.mean():.1f} | Desv. estandar: {gray.std():.1f}')
print(f'Pixeles oscuros (<64): {100*np.mean(gray < 64):.1f}%')

# Lectura: la masa esta en intensidades bajas -> imagen oscura.

In [ ]:
plt.figure(figsize=(8, 3))
for i, c, n in zip(range(3), ('b', 'g', 'r'), ('B', 'G', 'R')):
    plt.plot(histograma(img, i), color=c, label=n)
plt.title('Histogramas por canal - RGB'); plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3); plt.show()

plt.figure(figsize=(8, 3))
for i, c, n in zip(range(3), ('m', 'c', 'y'), ('H (Tono)', 'S (Sat.)', 'V (Brillo)')):
    plt.plot(histograma(hsv, i), color=c, label=n)
plt.title('Histogramas por canal - HSV'); plt.xlabel('Valor'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3); plt.show()

# Lectura: R, G, B con medias casi iguales -> escena acromatica.
# En HSV, S baja -> poca saturacion; V repite el patron de grises -> escena oscura.

---
## Ejercicio A.2 - Ecualizacion Global vs CLAHE

**Global** (`cv2.equalizeHist`): una sola CDF $s = (L-1) \cdot \mathrm{CDF}(r)$ para toda la imagen. Maximiza el contraste global pero la pendiente $\propto p(r)$ sobre-amplifica el ruido.

**CLAHE** (`cv2.createCLAHE`): divide la imagen en celdas (`tileGridSize`) y ecualiza cada una con su propia CDF local, recortando el histograma segun `clipLimit` para acotar la ganancia. Se prueba con `clipLimit` 2.0 y 4.0, y `tileGridSize` (8,8) y (16,16).


In [ ]:
eq_global = cv2.equalizeHist(gray)

configs = [(2.0, (8, 8)), (2.0, (16, 16)), (4.0, (8, 8)), (4.0, (16, 16))]
clahe_results = {}
for clip, tile in configs:
    c = cv2.createCLAHE(clipLimit=clip, tileGridSize=tile).apply(gray)
    clahe_results[f'clip={clip}, tile={tile}'] = c

imgs = [gray, eq_global] + list(clahe_results.values())
titulos = ['Original', 'Eq. global'] + list(clahe_results.keys())

plt.figure(figsize=(14, 8))
for i, (im, t) in enumerate(zip(imgs, titulos)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(im, cmap='gray', vmin=0, vmax=255)
    plt.title(t, fontsize=9); plt.axis('off')
plt.tight_layout(); plt.show()

# Comparacion de histogramas en dos plots para que se distingan bien
plt.figure(figsize=(13, 4))

plt.subplot(1, 2, 1)
plt.plot(histograma(gray), color='black', lw=2.5, label='Original')
plt.plot(histograma(eq_global), color='red', lw=2, ls='--', label='Eq. global')
plt.title('Original vs Ecualizacion global')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(); plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(histograma(gray), color='black', lw=2.5, label='Original')
plt.plot(histograma(clahe_results['clip=2.0, tile=(8, 8)']),
         color='tab:blue', lw=2, ls='-', label='CLAHE c=2.0 t=(8,8)')
plt.plot(histograma(clahe_results['clip=2.0, tile=(16, 16)']),
         color='tab:blue', lw=2, ls='--', label='CLAHE c=2.0 t=(16,16)')
plt.plot(histograma(clahe_results['clip=4.0, tile=(8, 8)']),
         color='tab:orange', lw=2, ls='-', label='CLAHE c=4.0 t=(8,8)')
plt.plot(histograma(clahe_results['clip=4.0, tile=(16, 16)']),
         color='tab:orange', lw=2, ls='--', label='CLAHE c=4.0 t=(16,16)')
plt.title('Original vs 4 configuraciones CLAHE')
plt.xlabel('Intensidad'); plt.ylabel('N. pixeles')
plt.xlim(0, 255); plt.legend(fontsize=8, loc='upper right'); plt.grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Lectura: la eq. global extiende el histograma a todo [0,255] pero granula el fondo.
# CLAHE con clipLimit bajo (2.0, azul) realza sin ese ruido.
# CLAHE con clipLimit alto (4.0, naranja) logra mas contraste pero el grano vuelve a notarse.

In [ ]:
# --- Cuadricula de CLAHE sobre la imagen ---
# Cada celda se ecualiza con SU PROPIA CDF local, no con la de toda la imagen.

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, tile in zip(axes, [(8, 8), (16, 16)]):
    overlay = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // tile[0], w // tile[1]

    for i in range(1, tile[0]):
        y = i * cell_h
        cv2.line(overlay, (0, y), (w, y), (0, 0, 255), 2)
    for j in range(1, tile[1]):
        x = j * cell_w
        cv2.line(overlay, (x, 0), (x, h), (0, 0, 255), 2)

    ax.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    ax.set_title(f'tileGridSize = {tile} -> {tile[0] * tile[1]} celdas')
    ax.axis('off')

plt.suptitle('Cada celda se ecualiza con su propia CDF local', y=1.02)
plt.tight_layout(); plt.show()

---
## Que pasa dentro de cada celda

Veamos 4 recortes de zonas distintas y como CLAHE los trata de forma independiente.


In [ ]:
h, w = img.shape[:2]
ps = 128

patches_pos = [
    ('Sup. izq.',  (slice(0, ps),        slice(0, ps))),
    ('Sup. der.',  (slice(0, ps),        slice(w - ps, w))),
    ('Inf. izq.',  (slice(h - ps, h),    slice(0, ps))),
    ('Inf. der.',  (slice(h - ps, h),    slice(w - ps, w))),
]

patches_gray = {nombre: cv2.cvtColor(img[y, x], cv2.COLOR_BGR2GRAY)
                 for nombre, (y, x) in patches_pos}

clahe_local = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
patches_eq = {nombre: clahe_local.apply(g) for nombre, g in patches_gray.items()}

nombres = list(patches_gray.keys())
colores = ['steelblue', 'seagreen', 'darkorange', 'mediumpurple']

fig, axes = plt.subplots(2, 4, figsize=(16, 6))

for i, nombre in enumerate(nombres):
    axes[0, i].imshow(cv2.cvtColor(img[patches_pos[i][1][0], patches_pos[i][1][1]],
                                     cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f'{nombre}\nantes')
    axes[0, i].axis('off')

    axes[1, i].imshow(patches_eq[nombre], cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title('despues CLAHE')
    axes[1, i].axis('off')

plt.suptitle('4 parches antes y despues de aplicar CLAHE localmente', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for color, nombre in zip(colores, nombres):
    axes[0].plot(histograma(patches_gray[nombre]), color=color,
                 label=nombre, alpha=0.85, lw=1.5)
    axes[1].plot(histograma(patches_eq[nombre]), color=color,
                 label=nombre, alpha=0.85, lw=1.5)

axes[0].set_title('Histogramas ANTES de CLAHE (4 parches superpuestos)')
axes[1].set_title('Histogramas DESPUES de CLAHE (4 parches superpuestos)')
for ax in axes:
    ax.set_xlabel('Intensidad'); ax.set_ylabel('N. pixeles')
    ax.set_xlim(0, 255); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

---
## Pregunta 1.1 - Sobre-amplificacion de ruido

> Por que la ecualizacion global arruina zonas homogeneas (pared lisa, fondo) y CLAHE no?

**Respuesta.** La ecualizacion global mira TODA la imagen y aplica una sola transformacion. Si una zona es muy homogenea (como las paredes lisas del fondo en `im1.png`), casi todos sus pixeles tienen la misma intensidad y el histograma ahi tiene un **pico muy alto**. La transformacion le dice a esa zona "estirate mucho", y entonces cualquier pequena diferencia entre pixeles (el ruido del sensor, de solo $\pm 2$ niveles, invisible en la foto original) se multiplica y aparece como **manchas visibles**.

Esto se ve claramente en las imagenes del Ejercicio A.2 y en la demo de los 4 parches:

- **Ecualizacion global:** las paredes del fondo y la niebla aparecen con **grano** (manchitas) que antes no estaban.
- **CLAHE con `clipLimit = 2.0`:** las mismas superficies quedan limpias, porque acota cuanto se puede estirar el histograma.
- **CLAHE con `clipLimit = 4.0`:** permite mas estiramiento y el grano vuelve a aparecer.

**CLAHE** resuelve esto con dos trucos que ya vimos en las visualizaciones:

1. **Recortar el histograma:** si una intensidad tiene demasiados pixeles, la cape a un maximo (`clipLimit`) y reparte el excedente a las demas intensidades. Asi nadie puede estirarse de mas. Esto es lo que hace que las paredes lisas no se granularicen.
2. **Trabajar por cuadritos:** como vimos en la cuadricula roja sobre la imagen, divide la imagen en celdas (`tileGridSize`) y ecualiza cada una con su propia estadistica. Asi el pico de la pared del fondo solo afecta a esa zona, no consume rango del resto. La demo de los 4 parches mostro que cada cuadrito se transforma de forma independiente.


---
### Como se representa un color en cada espacio

Para entender la Pregunta 1.2, primero veamos como **el mismo color** se describe en cada sistema. La siguiente animacion rota el tono (H) de 0 a 180 y muestra, en paralelo, los valores que toma ese color en:

- **RGB** (3 numeros entre 0 y 255)
- **HSV** (H entre 0-179, S y V entre 0-255)
- **CIELAB** (L entre 0-100, a y b entre -128 y 127)

Observa como al rotar el tono:

- En **RGB** los **tres** numeros cambian a la vez (porque el color se transforma).
- En **HSV** **solo cambia H**, mientras S y V quedan fijos (el brillo y la pureza no se tocan).
- En **CIELAB** cambian **a y b** (los ejes de color), mientras **L queda casi fijo** (mismo brillo).


In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

# Precomputar todas las representaciones para que la animacion sea fluida
n_frames = 90
rgb_frames = []
rgb_vals_list = []
lab_vals_list = []

for f in range(n_frames):
    H = int(f * 180 / n_frames)
    hsv_arr = np.uint8([[[H, 255, 255]]])  # S y V al maximo para ver el color puro
    rgb_arr = cv2.cvtColor(hsv_arr, cv2.COLOR_HSV2RGB)
    bgr_arr = cv2.cvtColor(rgb_arr, cv2.COLOR_RGB2BGR)
    lab_arr = cv2.cvtColor(bgr_arr, cv2.COLOR_BGR2Lab)

    rgb_frames.append(rgb_arr)
    rgb_vals_list.append(rgb_arr[0, 0].astype(int))
    lab_vals_list.append(lab_arr[0, 0].astype(int))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

def animate(frame):
    H = int(frame * 180 / n_frames)
    rgb_arr = rgb_frames[frame]
    rgb_vals = rgb_vals_list[frame]
    lab_vals = lab_vals_list[frame]

    # 1. Bloque de color
    axes[0].clear()
    axes[0].imshow(rgb_arr)
    axes[0].set_title(f'Color actual\nH = {H}', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    # 2. Barras RGB
    axes[1].clear()
    axes[1].bar(['R', 'G', 'B'], rgb_vals,
                 color=['red', 'green', 'blue'], alpha=0.85)
    axes[1].set_ylim(0, 255)
    axes[1].set_title(f'RGB\n({rgb_vals[0]}, {rgb_vals[1]}, {rgb_vals[2]})', fontsize=11)
    axes[1].set_ylabel('0 - 255')
    axes[1].grid(alpha=0.3, axis='y')

    # 3. Barras HSV (H, S, V)
    axes[2].clear()
    axes[2].bar(['H', 'S', 'V'], [H, 255, 255],
                 color=['gold', 'lightcyan', 'lightyellow'], edgecolor='black')
    axes[2].set_ylim(0, 260)
    axes[2].set_title(f'HSV\n(H={H}, S=255, V=255)', fontsize=11)
    axes[2].set_ylabel('H: 0-179, S,V: 0-255')
    axes[2].grid(alpha=0.3, axis='y')

    # 4. Barras CIELAB
    axes[3].clear()
    axes[3].bar(['L', 'a', 'b'], lab_vals,
                 color=['lightgray', 'lightgreen', 'lightblue'], edgecolor='black')
    axes[3].axhline(0, color='black', lw=0.5)
    axes[3].set_title(f'CIELAB\n(L={lab_vals[0]}, a={lab_vals[1]}, b={lab_vals[2]})', fontsize=11)
    axes[3].set_ylabel('L: 0-100, a,b: -128 a 127')
    axes[3].grid(alpha=0.3, axis='y')

anim = animation.FuncAnimation(fig, animate, frames=n_frames, interval=100)
plt.tight_layout()
plt.close()
HTML(anim.to_jshtml())


In [ ]:
# --- Atlas estatico: 8 colores del espectro con sus valores en cada espacio ---
# Equivalente a la version interactiva pero sin requerir ipywidgets.
# Cada fila muestra un color y como se representa en RGB, HSV y CIELAB.

hues = [0, 22, 45, 67, 90, 112, 135, 157]

fig, axes = plt.subplots(len(hues), 4, figsize=(14, 2.2 * len(hues)))

for i, H in enumerate(hues):
    hsv_arr = np.uint8([[[H, 255, 255]]])
    rgb_arr = cv2.cvtColor(hsv_arr, cv2.COLOR_HSV2RGB)
    bgr_arr = cv2.cvtColor(rgb_arr, cv2.COLOR_RGB2BGR)
    lab_arr = cv2.cvtColor(bgr_arr, cv2.COLOR_BGR2Lab)

    rgb_vals = rgb_arr[0, 0].astype(int)
    lab_vals = lab_arr[0, 0].astype(int)

    # 1. Bloque de color
    axes[i, 0].imshow(rgb_arr)
    axes[i, 0].set_title(f'Color (H = {H})', fontsize=10)
    axes[i, 0].axis('off')

    # 2. Barras RGB
    axes[i, 1].bar(['R', 'G', 'B'], rgb_vals,
                    color=['red', 'green', 'blue'], alpha=0.85)
    axes[i, 1].set_ylim(0, 255)
    axes[i, 1].set_title(f'RGB: ({rgb_vals[0]}, {rgb_vals[1]}, {rgb_vals[2]})', fontsize=9)
    axes[i, 1].grid(alpha=0.3, axis='y')

    # 3. Barras HSV
    axes[i, 2].bar(['H', 'S', 'V'], [H, 255, 255],
                    color=['gold', 'lightcyan', 'lightyellow'], edgecolor='black')
    axes[i, 2].set_ylim(0, 260)
    axes[i, 2].set_title(f'HSV: ({H}, 255, 255)', fontsize=9)
    axes[i, 2].grid(alpha=0.3, axis='y')

    # 4. Barras CIELAB
    axes[i, 3].bar(['L', 'a', 'b'], lab_vals,
                    color=['lightgray', 'lightgreen', 'lightblue'], edgecolor='black')
    axes[i, 3].axhline(0, color='black', lw=0.5)
    axes[i, 3].set_title(f'LAB: ({lab_vals[0]}, {lab_vals[1]}, {lab_vals[2]})', fontsize=9)
    axes[i, 3].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Conclusion: al comparar las filas se ve que en RGB los 3 canales cambian a la vez,
# en HSV solo cambia H (S y V quedan fijos), y en CIELAB cambian a y b (L queda casi fijo).
# Esto demuestra que HSV y CIELAB separan brillo de color, mientras que RGB los mezcla.

---
## Pregunta 1.2 - Canal de brillo vs RGB

> Por que es mala practica ecualizar R, G y B por separado y es mejor aplicar CLAHE solo sobre V (HSV) o L (CIELAB)?

**Respuesta.**

1. **El color depende de las proporciones R:G:B, no de sus valores absolutos.** Multiplicar los tres valores por la misma constante no cambia el color: $(100, 50, 50)$ y $(200, 100, 100)$ son el mismo rojo.

2. **Ecualizar R, G, B por separado rompe esas proporciones.** Cada canal se estira con su propia CDF, asi que el estiramiento es distinto para cada uno y las razones se alteran. Ejemplo: el pixel rojo $(100, 50, 50)$ puede terminar como $(200, 100, 60)$ -> ya no es el mismo rojo, sino un color distinto (mas naranja).

3. **HSV y CIELAB separan el brillo del color.** Guardan la informacion en dos partes:
   - **Luminancia** (V en HSV, L en CIELAB): brillo e iluminacion.
   - **Crominancia** (H, S en HSV; a, b en CIELAB): el color en si.

4. **CLAHE solo sobre V o L preserva el tono.** Redistribuye el brillo (donde esta el contraste) sin tocar las proporciones cromaticas, asi los colores originales se mantienen.

5. **CIELAB es mejor que HSV** porque es perceptualmente uniforme: cambios iguales de L se perciben igual en zonas oscuras y claras.

La celda siguiente lo demuestra empiricamente sobre `im1.png`: viraje de color con R,G,B independientes vs tono preservado con CLAHE en V o en L.


In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# (a) Mala practica: ecualizar B, G y R por separado
b, g, r = cv2.split(img)
eq_rgb = cv2.merge([cv2.equalizeHist(b), cv2.equalizeHist(g), cv2.equalizeHist(r)])

# (b) Buena practica: CLAHE solo sobre V (HSV)
h, s, v = cv2.split(hsv)
eq_hsv = cv2.cvtColor(cv2.merge([h, s, clahe.apply(v)]), cv2.COLOR_HSV2BGR)

# (c) Buena practica: CLAHE solo sobre L (CIELAB)
lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)
L, A, B = cv2.split(lab)
eq_lab = cv2.cvtColor(cv2.merge([clahe.apply(L), A, B]), cv2.COLOR_Lab2BGR)

casos = [img, eq_rgb, eq_hsv, eq_lab]
titulos = ['Original', 'Eq. R,G,B independiente (tonos falsos)', 'CLAHE solo en V (HSV)', 'CLAHE solo en L (CIELAB)']

plt.figure(figsize=(12, 9))
for i, (im, t) in enumerate(zip(casos, titulos)):
    plt.subplot(2, 2, i + 1)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.title(t, fontsize=10); plt.axis('off')
plt.tight_layout(); plt.show()

---
## Conclusiones del Modulo A

1. **Histograma como diagnostico.** El histograma es la PDF discreta de las intensidades de la imagen. Leerlo permite saber de un vistazo si la imagen es oscura, clara, o de contraste bajo antes de aplicar cualquier transformacion. En `im1.png` la masa en valores bajos indica sombras marcadas: caso ideal para ecualizar.

2. **Ecualizacion global: contraste maximo, ruido amplificado.** Usa una sola CDF para toda la imagen. Maximiza el contraste global pero, como la pendiente $ds/dr \propto p(r)$, cualquier pico en la PDF (zona homogenea) genera una pendiente enorme que amplifica el ruido de pocas unidades a decenas de niveles de salida. Resultado: zonas lisas granuladas.

3. **CLAHE: contraste local con ruido acotado.** Divide la imagen en celdas y aplica una CDF **local** con **clipping** del histograma segun `clipLimit`. Asi:
   - `clipLimit` controla la ganancia maxima (contraste vs ruido).
   - `tileGridSize` controla la escala de adaptacion (local vs global).
   Cada celda se ecualiza con SU propia estadistica, asi una zona oscura se realza segun su CDF, no segun la CDF de la imagen entera.

4. **Color: siempre sobre la luminancia.** En imagenes a color la ecualizacion debe operarse sobre el canal de luminancia (V en HSV o L en CIELAB), nunca sobre R, G, B por separado. Ecualizar canales independientes rompe las proporciones cromaticas que definen el tono y produce colores falsos. CIELAB es la mejor opcion cuando se busca fidelidad perceptual.


---
# Modulo B: Espacios de Color y Segmentacion (7.5%)

Imagen de trabajo: `im2.png` (con marcadores de varios colores). Aqui desacoplamos la crominancia (color) de la luminancia (brillo) y comparamos segmentacion en RGB vs HSV.


In [ ]:
img2 = cv2.imread('im2.png')
assert img2 is not None, 'No se encontro im2.png'
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
hsv2 = cv2.cvtColor(img2, cv2.COLOR_BGR2HSV)
lab2 = cv2.cvtColor(img2, cv2.COLOR_BGR2Lab)
print('Imagen:', img2.shape)
mostrar(img2, 'Imagen original im2.png')


---
## Ejercicio B.1 - Descomposicion de Canales

**Que pide:** tomar la imagen y separar cada uno de sus componentes, mostrandolos uno a uno como imagen gris en vez de mezclados en color.

**Por que:** una imagen a color guarda 3 numeros por pixel. Si los miras juntos ves color; si miras UNO a la vez ves una historia distinta (solo brillo, solo tono, solo pureza, etc).

**Que veremos:**

| Imagen | Que muestra |
|---|---|
| Grises | Cuanto brillo hay en general |
| H | Que color tiene cada pixel (en zonas grises es ruido) |
| S | Que tan puro/vivo es el color |
| V | Brillo (parecido a grises) |
| L | Brillo perceptual (similar a V) |
| a | Eje rojo ↔ verde |
| b | Eje amarillo ↔ azul |

**Como leer cada canal:**

- **H:** zonas con color vivo aparecen con valores; zonas grises = ruido.
- **S:** zonas blancas/negras = sin color; zonas oscuras = colores vivos.
- **V:** replica el patron de grises (intensidad).
- **L:** similar a V pero perceptual.
- **a:** blanco = rojizo, negro = verdoso, gris = neutro.
- **b:** blanco = amarillento, negro = azulado, gris = neutro.


In [ ]:
# Escala de grises
mostrar(gray2, 'Escala de grises', cmap='gray')

# HSV: H (Tono), S (Saturacion), V (Brillo)
canales_hsv = [('H (Tono)', hsv2[:, :, 0]),
               ('S (Sat.)', hsv2[:, :, 1]),
               ('V (Brillo)', hsv2[:, :, 2])]
for nombre, canal in canales_hsv:
    mostrar(canal, nombre, cmap='gray')

# CIELAB: L (Luminancia), a (verde-rojo), b (azul-amarillo)
canales_lab = [('L (Luminancia)', lab2[:, :, 0]),
               ('a (verde-rojo)', lab2[:, :, 1]),
               ('b (azul-amarillo)', lab2[:, :, 2])]
for nombre, canal in canales_lab:
    mostrar(canal, nombre, cmap='gray')

# Lectura:
# - H: solo aparece donde hay color saturado; en zonas grises H es ruido.
# - S: zonas blancas/negras = 0; zonas con colores vivos = alto.
# - V: replica el patron del histograma de grises (intensidad).
# - L: similar a V pero perceptual.
# - a: positivo = rojizo, negativo = verdoso.
# - b: positivo = amarillento, negativo = azulado.

---
## Ejercicio B.2 - Segmentacion RGB vs HSV

Segmentamos un marcador de color con `cv2.inRange` definiendo rangos en RGB y en HSV. Luego simulamos una sombra sobre la imagen y comparamos que metodo mantiene el marcador.


In [ ]:
# Que estamos haciendo aqui:
# Para CADA pixel de la imagen preguntamos: '¿este pixel es ROJO?'
#   - Si la respuesta es SI  -> el pixel se pinta BLANCO (255) en la mascara.
#   - Si la respuesta es NO  -> el pixel se pinta NEGRO (0) en la mascara.
# Resultado: una imagen blanco/negro donde solo aparece el marcador rojo.

# Suponemos que el marcador es ROJO. Ajusta los rangos segun el color
# real de tus marcadores en im2.png.

# --- Mascara en RGB (rangos en B, G, R) ---
# Acepta el pixel solo si las 3 condiciones se cumplen:
#   R entre 150 y 255 (rojo alto)
#   G entre 0 y 80    (verde bajo)
#   B entre 0 y 80    (azul bajo)
lower_rgb = np.array([0, 0, 150])
upper_rgb = np.array([80, 80, 255])
mask_rgb = cv2.inRange(img2, lower_rgb, upper_rgb)

# --- Mascara en HSV (rangos en H, S, V) ---
# Acepta el pixel si su tono cae en alguna de las dos zonas del rojo:
#   H entre 0 y 10      O     H entre 170 y 179
# (rojo envuelve el circulo de tonos: esta al inicio y al final)
# Ademas: S entre 100 y 255 (saturacion media-alta) y V entre 50 y 255 (no muy oscuro).
lower_hsv1 = np.array([0, 100, 50])
upper_hsv1 = np.array([10, 255, 255])
lower_hsv2 = np.array([170, 100, 50])
upper_hsv2 = np.array([179, 255, 255])
mask_hsv = cv2.inRange(hsv2, lower_hsv1, upper_hsv1) | cv2.inRange(hsv2, lower_hsv2, upper_hsv2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(mask_rgb, cmap='gray'); axes[0].set_title('Mascara RGB'); axes[0].axis('off')
axes[1].imshow(mask_hsv, cmap='gray'); axes[1].set_title('Mascara HSV'); axes[1].axis('off')
plt.tight_layout(); plt.show()

# Cuantos pixeles 'rojos' detecto cada metodo?
# Como cada pixel detectado vale 255, dividir la suma entre 255 da el conteo.
print(f'Pixeles detectados RGB: {mask_rgb.sum() / 255:.0f}')
print(f'Pixeles detectados HSV: {mask_hsv.sum() / 255:.0f}')


In [ ]:
# --- Simulacion de sombra: oscurecer la imagen multiplicando V ---
hsv_shadow = hsv2.copy()
hsv_shadow[:, :, 2] = (hsv_shadow[:, :, 2] * 0.4).astype(np.uint8)  # V al 40%
img_shadow = cv2.cvtColor(hsv_shadow, cv2.COLOR_HSV2BGR)

# Aplicar las mismas mascaras (rangos fijos) a la imagen con sombra
mask_rgb_shadow = cv2.inRange(img_shadow, lower_rgb, upper_rgb)
mask_hsv_shadow = cv2.inRange(hsv_shadow, lower_hsv1, upper_hsv1) | cv2.inRange(hsv_shadow, lower_hsv2, upper_hsv2)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].imshow(cv2.cvtColor(img_shadow, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title('Imagen con sombra simulada'); axes[0, 0].axis('off')
axes[0, 1].imshow(mask_rgb_shadow, cmap='gray'); axes[0, 1].set_title('Mascara RGB bajo sombra'); axes[0, 1].axis('off')
axes[1, 0].imshow(mask_hsv_shadow, cmap='gray'); axes[1, 0].set_title('Mascara HSV bajo sombra'); axes[1, 0].axis('off')
axes[1, 1].imshow(mask_hsv_shadow - mask_rgb_shadow, cmap='RdBu', vmin=-255, vmax=255)
axes[1, 1].set_title('Diferencia HSV - RGB\n(azul = HSV detecta mas)'); axes[1, 1].axis('off')
plt.tight_layout(); plt.show()

# Conclusion: bajo la sombra simulada, la mascara RGB pierde pixeles del marcador
# (porque el rango R, G, B fijos ya no cubre al rojo oscuro).
# La mascara HSV sigue funcionando porque H es invariante al brillo.

---
## Pregunta 1.3 - Invarianza del Matiz

> Por que el canal H (Hue / Matiz) en el espacio HSV ofrece inmunidad frente a sombras y cambios moderados de iluminacion en comparacion con el espacio RGB?

**Respuesta.** Porque H se calcula a partir de las **proporciones** entre R, G y B, no de sus valores absolutos. Una sombra reduce los tres canales de forma proporcional (la imagen se vuelve mas oscura pero sigue siendo del mismo color), asi que las razones R:G:B apenas cambian y H permanece casi constante.

En cambio, una mascara RGB depende de los valores absolutos de cada canal. Si el umbral dice "R entre 150 y 255", una sombra que baja R de 200 a 80 deja al pixel fuera del rango aunque visualmente siga siendo "el mismo rojo". Por eso la segmentacion RGB es fragil ante sombras y HSV no.


---
## Pregunta 1.4 - Indeterminacion en HSV

> En que zonas del espacio de color el canal H se vuelve matematicamente indeterminado o ruidoso?

**Respuesta.** En dos casos:

1. **Cuando S tiende a 0** (color desaturado, casi gris): los tres canales R, G, B son casi iguales, asi que la proporcion R:G:B es ~1:1:1. Matematicamente eso hace que H sea ambiguo y pequenas variaciones de ruido hacen que H salte a cualquier valor.

2. **Cuando V tiende a 0** (color muy oscuro, casi negro): los tres canales son casi cero, divididos por numeros pequenos, las proporciones son ruido puro. H no representa nada.

Por eso las zonas blancas, negras o grises de la imagen muestran H "saltar" o dar cualquier color. Es una limitacion conocida de HSV.


---
# Modulo D: Morfologia Matematica y Elementos Estructurantes (10%)

La morfologia matematica limpia mascaras binarias (blanco/negro) usando un **kernel** (una ventanita geometrica que se desliza por la imagen). Cada operacion decide que pixeles sobreviven segun la forma del kernel.

Para los ejercicios generamos una imagen sintetica limpia con 3 formas (circulo, cuadrado, linea) y dos versiones con ruido: una con **ruido sal** (puntos blancos en el fondo) y otra con **ruido pimienta** (huecos negros en los objetos).


In [ ]:
# --- Generar imagenes sinteticas para los ejercicios de morfologia ---
np.random.seed(42)

# Mascara limpia: circulo + cuadrado + linea delgada
mask_clean = np.zeros((200, 300), dtype=np.uint8)
cv2.circle(mask_clean, (70, 70), 35, 255, -1)                  # circulo
cv2.rectangle(mask_clean, (150, 40), (240, 110), 255, -1)      # cuadrado
cv2.line(mask_clean, (40, 160), (260, 170), 255, 4)            # linea delgada (simula varilla)

# Version con ruido SAL: puntos blancos aleatorios en el fondo
mask_salt = mask_clean.copy()
salt_noise = np.random.random(mask_clean.shape) < 0.03
mask_salt[salt_noise] = 255

# Version con ruido PIMIENTA: huecos negros aleatorios en los objetos
mask_pepper = mask_clean.copy()
pepper_noise = np.random.random(mask_clean.shape) < 0.05
mask_pepper[pepper_noise & (mask_pepper == 255)] = 0

# Version dedicada para Black-Hat: fondo blanco con detalles negros visibles
# (circulos pequenos, rayitas, huecos - como defectos reales que querriamos detectar)
test_blackhat = np.ones_like(mask_clean) * 255  # fondo blanco

# Circulos pequenos (defectos circulares) - tuplas (cx, cy, radio)
for cx, cy, r in [(40, 40, 4), (130, 50, 5), (220, 50, 3), (60, 150, 6), (200, 170, 4)]:
    cv2.circle(test_blackhat, (cx, cy), r, 0, -1)

# Rayitas pequenas (defectos lineales)
cv2.line(test_blackhat, (90, 30), (110, 30), 0, 2)
cv2.line(test_blackhat, (170, 130), (200, 130), 0, 2)
cv2.line(test_blackhat, (40, 100), (40, 120), 0, 2)

# Cuadradito pequeno (otro tipo de defecto)
cv2.rectangle(test_blackhat, (250, 100), (270, 115), 0, -1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(mask_clean, cmap='gray'); axes[0].set_title('Limpia'); axes[0].axis('off')
axes[1].imshow(mask_salt, cmap='gray'); axes[1].set_title('Ruido SAL'); axes[1].axis('off')
axes[2].imshow(mask_pepper, cmap='gray'); axes[2].set_title('Ruido PIMIENTA'); axes[2].axis('off')
axes[3].imshow(test_blackhat, cmap='gray'); axes[3].set_title('Test Black-Hat\n(fondo blanco + defectos)'); axes[3].axis('off')
plt.tight_layout(); plt.show()


---
## Ejercicio D.1 - Erosion y Dilatacion

- **Erosion:** encoge los objetos blancos (borra motas pequenas y hace los objetos mas finos).
- **Dilatacion:** ensancha los objetos blancos (rellena huecos y hace los objetos mas gruesos).

Probamos 1, 2 y 4 iteraciones para ver como se acumula el efecto.


In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, iters in enumerate([1, 2, 4]):
    eroded = cv2.erode(mask_salt, kernel, iterations=iters)
    dilated = cv2.dilate(mask_pepper, kernel, iterations=iters)

    axes[i, 0].imshow(mask_salt if i == 0 else np.zeros_like(mask_salt), cmap='gray')
    axes[i, 0].set_title('Original sal' if i == 0 else ''); axes[i, 0].axis('off')
    axes[i, 1].imshow(eroded, cmap='gray'); axes[i, 1].set_title(f'Erosion {iters} iter.'); axes[i, 1].axis('off')
    axes[i, 2].imshow(mask_pepper if i == 0 else np.zeros_like(mask_pepper), cmap='gray')
    axes[i, 2].set_title('Original pimienta' if i == 0 else ''); axes[i, 2].axis('off')
    axes[i, 3].imshow(dilated, cmap='gray'); axes[i, 3].set_title(f'Dilatacion {iters} iter.'); axes[i, 3].axis('off')

plt.suptitle('Erosion sobre ruido sal | Dilatacion sobre ruido pimienta', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()

# Conclusion:
# - Erosion: con mas iteraciones, el ruido sal desaparece pero los objetos se encogen.
# - Dilatacion: con mas iteraciones, los huecos se rellenan pero los objetos engordan.

---
## Ejercicio D.2 - Apertura y Cierre

- **Apertura (Opening) = Erosion + Dilatacion:** elimina motas blancas aisladas (ruido sal) sin deformar mucho el objeto.
- **Cierre (Closing) = Dilatacion + Erosion:** rellena huecos negros internos (ruido pimienta) sin deformar mucho el objeto.


In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

opening_salt = cv2.morphologyEx(mask_salt, cv2.MORPH_OPEN, kernel)
closing_pepper = cv2.morphologyEx(mask_pepper, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes[0, 0].imshow(mask_salt, cmap='gray'); axes[0, 0].set_title('Original con ruido SAL'); axes[0, 0].axis('off')
axes[0, 1].imshow(opening_salt, cmap='gray'); axes[0, 1].set_title('Apertura (MORPH_OPEN)'); axes[0, 1].axis('off')
axes[0, 2].imshow(mask_salt - opening_salt, cmap='gray'); axes[0, 2].set_title('Lo que elimino la apertura'); axes[0, 2].axis('off')

axes[1, 0].imshow(mask_pepper, cmap='gray'); axes[1, 0].set_title('Original con ruido PIMIENTA'); axes[1, 0].axis('off')
axes[1, 1].imshow(closing_pepper, cmap='gray'); axes[1, 1].set_title('Cierre (MORPH_CLOSE)'); axes[1, 1].axis('off')
axes[1, 2].imshow(closing_pepper - mask_pepper, cmap='gray'); axes[1, 2].set_title('Lo que relleno el cierre'); axes[1, 2].axis('off')

plt.tight_layout(); plt.show()

# Conclusion:
# - Apertura elimina las motas blancas pequenas del fondo sin deformar visiblemente los objetos.
# - Cierre rellena los huecos negros pequenos dentro de los objetos sin deformar visiblemente los bordes.

---
## Ejercicio D.3 - Morfologia Avanzada

- **Gradiente morfologico = Dilatacion - Erosion:** muestra solo los **bordes** de los objetos.
- **Top-Hat (sombrero blanco) = Original - Apertura:** resalta **detalles brillantes pequenos** sobre **fondo oscuro**.
- **Black-Hat (sombrero negro) = Cierre - Original:** resalta **detalles oscuros pequenos** sobre **fondo claro**.

Cada operacion se aplica sobre una imagen de prueba adecuada:

| Operacion | Entrada | Que muestra |
|---|---|---|
| Gradiente | `mask_clean` (objetos grandes) | Solo los bordes (anillos) |
| Top-Hat | `mask_salt` (objetos + motas blancas) | Solo las motas blancas pequenas |
| Black-Hat | `test_blackhat` (fondo blanco + puntos negros) | Solo los puntos negros pequenos |


In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

gradiente = cv2.morphologyEx(mask_clean, cv2.MORPH_GRADIENT, kernel)
tophat = cv2.morphologyEx(mask_salt, cv2.MORPH_TOPHAT, kernel)
blackhat = cv2.morphologyEx(test_blackhat, cv2.MORPH_BLACKHAT, kernel)
blackhat_extraido = cv2.subtract(cv2.morphologyEx(test_blackhat, cv2.MORPH_CLOSE, kernel), test_blackhat)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))

# Fila 1: Gradiente
axes[0, 0].imshow(mask_clean, cmap='gray'); axes[0, 0].set_title('Entrada: limpia'); axes[0, 0].axis('off')
axes[0, 1].imshow(gradiente, cmap='gray'); axes[0, 1].set_title('Gradiente\n(bordes)'); axes[0, 1].axis('off')
axes[0, 2].axis('off'); axes[0, 3].axis('off')

# Fila 2: Top-Hat
axes[1, 0].imshow(mask_salt, cmap='gray'); axes[1, 0].set_title('Entrada: ruido SAL'); axes[1, 0].axis('off')
axes[1, 1].imshow(tophat, cmap='gray'); axes[1, 1].set_title('Top-Hat\n(motas blancas extraidas)'); axes[1, 1].axis('off')
axes[1, 2].imshow(np.maximum(mask_salt - cv2.morphologyEx(mask_salt, cv2.MORPH_OPEN, kernel), 0),
                   cmap='gray')
axes[1, 2].set_title('Sobre la entrada SAL\n(que elimino la apertura)'); axes[1, 2].axis('off')
axes[1, 3].axis('off')

# Fila 3: Black-Hat
axes[2, 0].imshow(test_blackhat, cmap='gray'); axes[2, 0].set_title('Entrada: test_blackhat'); axes[2, 0].axis('off')
axes[2, 1].imshow(blackhat, cmap='gray'); axes[2, 1].set_title('Black-Hat\n(defectos negros extraidos)'); axes[2, 1].axis('off')
axes[2, 2].imshow(blackhat_extraido, cmap='gray'); axes[2, 2].set_title('Sobre la entrada\n(que relleno el cierre)'); axes[2, 2].axis('off')
axes[2, 3].axis('off')

plt.tight_layout(); plt.show()

# Conclusion:
# - Gradiente: solo los bordes de los objetos (anillos finos).
# - Top-Hat: aparecen como puntos blancos las motas pequenas del ruido sal.
# - Black-Hat: aparecen como puntos blancos los detalles oscuros pequenos sobre fondo blanco
#   (los circulos, rayitas y cuadrados negros que pusimos en test_blackhat).

---
## Ejercicio D.4 - Estudio de Kernels

Comparamos 3 formas de kernel (`MORPH_RECT`, `MORPH_ELLIPSE`, `MORPH_CROSS`) en 4 tamanos (3x3, 5x5, 9x9, 15x15) sobre la mascara con ruido sal. La operacion usada es apertura (MORPH_OPEN).


In [ ]:
formas = {'Rectangular': cv2.MORPH_RECT, 'Eliptica': cv2.MORPH_ELLIPSE, 'Cruz': cv2.MORPH_CROSS}
tamanos = [3, 5, 9, 15]

# --- Figura A: forma de cada kernel (asi se ve cada uno antes de aplicarlo) ---
fig, axes = plt.subplots(len(formas), len(tamanos), figsize=(12, 7))

for i, (nombre, forma) in enumerate(formas.items()):
    for j, sz in enumerate(tamanos):
        k = cv2.getStructuringElement(forma, (sz, sz))
        axes[i, j].imshow(k, cmap='gray', vmin=0, vmax=255)
        if i == 0:
            axes[i, j].set_title(f'{sz}x{sz}', fontsize=11)
        if j == 0:
            axes[i, j].set_ylabel(nombre, fontsize=11, rotation=0, labelpad=50, va='center')
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

plt.suptitle('Figura A: FORMA de cada kernel (lo que se aplica)', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# --- Figura B: resultado de aplicar cada kernel con apertura ---
fig, axes = plt.subplots(len(formas), len(tamanos), figsize=(12, 7))

for i, (nombre, forma) in enumerate(formas.items()):
    for j, sz in enumerate(tamanos):
        k = cv2.getStructuringElement(forma, (sz, sz))
        cleaned = cv2.morphologyEx(mask_salt, cv2.MORPH_OPEN, k)
        axes[i, j].imshow(cleaned, cmap='gray')
        if i == 0:
            axes[i, j].set_title(f'{sz}x{sz}', fontsize=11)
        if j == 0:
            axes[i, j].set_ylabel(nombre, fontsize=11, rotation=0, labelpad=50, va='center')
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

plt.suptitle('Figura B: RESULTADO de apertura (MORPH_OPEN) con cada kernel', y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Conclusion (compara las celdas correspondientes entre Figura A y Figura B):
# - Rectangular: agresivo, esquinas marcadas. Limpia rapido pero deforma las esquinas del cuadrado.
# - Eliptica: respeta mejor formas circulares. Recomendado para esferas (marcador del pendulo).
# - Cruz: minimal, solo afecta 4 direcciones. Deja pasar mas detalle pero limpia menos.
# - Tamanos 3x3: apenas limpian, dejan ruido.
# - Tamanos 15x15: limpian bien pero pueden borrar objetos delgados (la linea delgada desaparece).

---
## Pregunta 1.6 - Kernel Eliptico vs Rectangular

> Por que para segmentar y calcular el centroide de un marcador esferico circular es mucho mas preciso usar un elemento estructurante eliptico que uno rectangular?

**Respuesta.** El kernel rectangular tiene **esquinas** que se proyectan mas alla del circulo real del marcador: cuando erosionas con el, los bordes del circulo se cortan en angulos rectos y el resultado es una forma mas cuadrada. Esto desplaza el centroide calculado hacia adentro del circulo original.

El kernel **eliptico** sigue la curvatura natural del circulo: erosionas uniformemente desde todos los angulos, asi la forma se mantiene circular y el centroide calculado queda mas cerca del centro real del marcador.

Para nuestro pendulo, el marcador es una esfera pequena: usar `MORPH_ELLIPSE` preserva su forma y el centroide es mas preciso, lo que se traduce en menos error al medir la trayectoria.


---
## Pregunta 1.7 - Reconexion de Varillas Delgadas

> Si durante el seguimiento del pendulo la cuerda o varilla delgada sufre discontinuidades (pequenas roturas de 2 a 4 pixeles) por reflejos, que operacion morfolgica y que tipo/tamano de kernel recomendaria para reconectarla sin deformar la masa?

**Respuesta.** Usar **Cierre** (`cv2.MORPH_CLOSE`) con un kernel **lineal rectangular** (mucho mas largo que ancho), por ejemplo `(N, 1)` o `(1, N)` donde N es la longitud de la rotura mas grande esperada (4 pixeles). El rectangulo debe orientarse en la direccion de la cuerda.

Por que funciona: el cierre primero dilata (tapando los huecos de 2-4 pixeles) y luego erosiona (restaurando el grosor original). Como el kernel es lineal en una sola direccion, la **masa circular** apenas se ve afectada (la dilatacion/erosion solo se aplica en la direccion de la cuerda).

Tamanos razonables: para roturas de hasta 4 pixeles, un kernel de `5x1` o `7x1` funciona bien. Mas alla, mejor cambiar a un kernel cuadrado.


---
## Pregunta 1.8 - Aplicaciones de Top-Hat y Black-Hat

> En que escenarios industriales o biomedicos resulta fundamental el uso de las transformadas Top-Hat y Black-Hat?

**Respuesta.**

**Top-Hat (resalta detalles brillantes pequenos):**
- **Inspeccion de PCB (placas electronicas):** detectar **pistas conductoras** finas (claras) sobre el fondo oscuro de la placa. Si una pista esta rota, el top-hat muestra la discontinuidad.
- **Inspeccion de superficies:** detectar rayones o marcas claras pequenas sobre un fondo uniforme.
- **Biomedicina:** resaltar **vasos sanguineos** en imagenes de retina o angiografias.

**Black-Hat (resalta detalles oscuros pequenos):**
- **Inspeccion de PCB:** detectar **defectos oscuros** (cortocircuitos, quemadura) sobre la superficie clara.
- **Control de calidad:** encontrar grietas o manchas oscuras en piezas metalicas o ceramicas.
- **Biomedicina:** detectar **lesiones pequenas o manchas** en imagenes de piel, rayos X, etc.

**Idea comun:** ambas operaciones extraen detalles pequenos que estan **sobre o dentro de un fondo uniforme**. Top-Hat para lo brillante, Black-Hat para lo oscuro.
